# indian_equities_daily_strat_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (135).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 27 |
| Detected functions | calculate_daily_correlation, calculate_daily_correlation, compute_position_sizes, precompute_rolling_statistics, run_backtest, calculate_max_drawdown, simulate_cumulative_pnl, save_plot_as_image, plot_cumulative_pnl_and_drawdown, plot_drawdown, plot_pnl_distribution, plot_monthly_profit_heatmap |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
# Now you can load the entire rows with ICICIBANK.NSE if required
BuyPrice_Stock1 = pd.read_parquet('ICICIBANK_2024/ICICIBANK_2024.parquet')
BuyPrice_Stock1

In [ ]:
import pandas as pd
BuyPrice_Stock2 = pd.read_parquet('LT_2024/LT_2024.parquet')
BuyPrice_Stock2

In [ ]:
import pandas as pd

# Assuming df1 and df2 are your two DataFrames and 'Date_Time' is the common column
combined_df = pd.merge(BuyPrice_Stock1, BuyPrice_Stock2, on='Date_Time', how='inner')  # 'inner' join will keep only the rows where 'Date_Time' is common in both datasets
combined_df.head()


In [ ]:
import pandas as pd

# Assuming combined_df is already defined and 'data' DataFrame is created
data = pd.DataFrame({
    'DateTime': combined_df['Date_Time'],
    'BuyPrice_Stock1': combined_df['BuyPrice_y'],
    'BuyPrice_Stock2': combined_df['BuyPrice_x'],
    'SellPrice_Stock1': combined_df['SellPrice_y'],
    'SellPrice_Stock2': combined_df['SellPrice_x']
})

# Remove rows where any price column has a 0 value
data = data[(data[['BuyPrice_Stock1', 'BuyPrice_Stock2', 'SellPrice_Stock1', 'SellPrice_Stock2']] != 0).all(axis=1)]

# Display the resulting DataFrame
data


In [ ]:
import pandas as pd

# Sample data structure
# data = pd.DataFrame({
#     'datetime': [...], 
#     'BuyPrice_Stock1': [...], 
#     'SellPrice_Stock1': [...], 
#     'BuyPrice_Stock2': [...], 
#     'SellPrice_Stock2': [...]
# })

# Ensure the 'datetime' column is in datetime format
data['datetime'] = pd.to_datetime(data['DateTime'])

# Extract the date directly for grouping
data['date'] = data['datetime'].dt.date

# Function to calculate daily correlations for a specific price type
def calculate_daily_correlation(df, price_type1, price_type2):
    return (
        df.groupby('date')[[price_type1, price_type2]]
        .corr()
        .iloc[0::2, -1]
        .reset_index(level=1, drop=True)
    )

# Calculate correlations for BuyPrice and SellPrice
daily_corr_buy = calculate_daily_correlation(data, 'BuyPrice_Stock1', 'BuyPrice_Stock2')
daily_corr_sell = calculate_daily_correlation(data, 'SellPrice_Stock1', 'SellPrice_Stock2')

# Combine results into a single DataFrame
daily_correlation = pd.DataFrame({
    'date': daily_corr_buy.index,
    'BuyPrice_correlation': daily_corr_buy.values,
    'SellPrice_correlation': daily_corr_sell.values
}).reset_index(drop=True)

daily_correlation.to_csv('daily_correlation.csv')

print(daily_correlation)

In [ ]:
# Ensure 'DateTime' is in datetime format
data['DateTime'] = pd.to_datetime(data['DateTime'])

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(data['DateTime'], data['BuyPrice_Stock1'], label='Buy Price Stock1', color='green')
plt.plot(data['DateTime'], data['BuyPrice_Stock2'], label='Buy Price Stock2', color='red')


# Formatting the plot
plt.xlabel('DateTime')
plt.ylabel('Price')
plt.title('Buy and Sell Prices of Stock1 and Stock2')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.grid(True)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create the figure
fig = plt.figure(figsize=(13, 7))

# Calculate the difference between the 'close' prices of BuyPrice_Stock1.NS and BuyPrice_Stock2.NS
price_difference_Buy = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])

# Plot the difference
plt.plot(price_difference_Buy, label='Buy Price Difference')

plt.legend()
plt.title("Difference in Prices: Stock2 - Stock1")
plt.xlabel("Date")
plt.ylabel("Price Difference")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Calculate the ratio of HdataC to ICICI
ratio = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
ratio_mean = ratio.rolling(window=5500).mean()
# Create the figure and plot the ratio
fig = plt.figure(figsize=(13, 7))
plt.plot(ratio, label='BuyPrice_Stock1 / BuyPrice_Stock2')

# Add a horizontal line at the mean of the ratio
plt.plot(ratio_mean, color='red', linestyle='--', label='Mean Ratio')

# Add legend, title, and labels
plt.legend()
plt.title("Ratio of BuyPrice_Stock1 to BuyPrice_Stock2 close Prices")
plt.xlabel("Date")
plt.ylabel("Ratio")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate the ratio of BuyPrice_Stock1 to BuyPrice_Stock2
ratio = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])

# Calculate the 43-min moving average
moving_average_43 = ratio.rolling(window=6900).mean()

# Calculate the mean and standard deviation of the ratio
mean_ratio = ratio.mean()
std_ratio = ratio.std()

# Calculate the Z-score using the ratio
z_score = (ratio - moving_average_43) / moving_average_43.std()

# Create the figure and plot the Z-score
fig = plt.figure(figsize=(13, 7))
plt.plot(z_score, label='Z-Score of BuyPrice_Stock1 / BuyPrice_Stock2', color='purple')

# Add horizontal lines for Z-scores of -1 and 1 for reference
plt.axhline(0.3, color='green', linestyle='--', label='Z-Score = 0.3')
plt.axhline(-0.3, color='green', linestyle='--', label='Z-Score = -0.3')


# Add buy and sell markers
buy_signals = z_score[z_score < -0.3]
sell_signals = z_score[z_score > 0.3]

# Mark buy signals
plt.scatter(buy_signals.index, buy_signals, marker='^', color='green', label='Buy Signal', s=100)
# Mark sell signals
plt.scatter(sell_signals.index, sell_signals, marker='v', color='red', label='Sell Signal', s=100)

# Add legend, title, and labels
plt.legend()
plt.title("Z-Score of BuyPrice_Stock1 to BuyPrice_Stock2 Ratio with Buy/Sell Signals")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.show()


In [ ]:
data['ratio'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
data['difference'] = data['BuyPrice_Stock1'] - data['BuyPrice_Stock2']
data['z_score_pair'] = (ratio - moving_average_43) / moving_average_43.std()
data

In [ ]:
# Initialize positions
positions = pd.Series(index=data.index, data=0)

# Generate buy signals (when z-score is less than -0.05)
positions[data['z_score_pair'] < -0.2] = 1  # Buy position

# Generate sell signals (when z-score is greater than 0.05)
positions[data['z_score_pair'] > 0.2] = -1  # Sell position

# Close all positions (when z-score is equal to 0)
positions[data['z_score_pair'] == 0] = 0  # Close all positions

# Add the positions to the DataFrame
data['position'] = positions

# Display the resulting DataFrame
data

In [ ]:
data.columns

In [ ]:
import pandas as pd

# Ensure 'datetime' column is in datetime format
data['datetime'] = pd.to_datetime(data['datetime'])

# Group by minute and count the number of ticks in each minute
ticks_per_minute = data.groupby(data['datetime'].dt.floor('T')).size()

# Calculate the average ticks per minute
average_ticks_per_minute = ticks_per_minute.mean()

print(f"Average ticks per minute: {average_ticks_per_minute}")


In [ ]:
import pandas as pd

# Sample data structure
# data = pd.DataFrame({
#     'datetime': [...], 
#     'BuyPrice_Stock1': [...], 
#     'SellPrice_Stock1': [...], 
#     'BuyPrice_Stock2': [...], 
#     'SellPrice_Stock2': [...]
# })

# Ensure the 'datetime' column is in datetime format
data['datetime'] = pd.to_datetime(data['DateTime'])

# Extract the date directly for grouping
data['date'] = data['datetime'].dt.date

# Filter out Fridays (where the weekday is 4)
data = data[data['datetime'].dt.weekday != 4]

# Function to calculate daily correlations for a specific price type
def calculate_daily_correlation(df, price_type1, price_type2):
    return (
        df.groupby('date')[[price_type1, price_type2]]
        .corr()
        .iloc[0::2, -1]
        .reset_index(level=1, drop=True)
    )

# Calculate correlations for BuyPrice and SellPrice
daily_corr_buy = calculate_daily_correlation(data, 'BuyPrice_Stock1', 'BuyPrice_Stock2')
daily_corr_sell = calculate_daily_correlation(data, 'SellPrice_Stock1', 'SellPrice_Stock2')

# Combine results into a single DataFrame
daily_correlation = pd.DataFrame({
    'date': daily_corr_buy.index,
    'BuyPrice_correlation': daily_corr_buy.values,
    'SellPrice_correlation': daily_corr_sell.values
}).reset_index(drop=True)

print(daily_correlation)


In [ ]:
import pandas as pd
import numpy as np

# Load your dataset
data['datetime'] = pd.to_datetime(data['DateTime'])
data.set_index('datetime', inplace=True)  # Set datetime as index if not already

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade
ma_window = 6700  # Moving average window
z_score_threshold = 0.2  # Z-score threshold

# Initialize variables
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
cumulative_pnl = 0
tradebook = []

# Vectorized position size calculation
data['max_risk'] = portfolio_value * risk_percentage
data['position_size1_buy'] = (data['max_risk'] / (2 * data['BuyPrice_Stock1'])).round()
data['position_size1_sell'] = (data['max_risk'] / (2 * data['SellPrice_Stock1'])).round()
data['position_size2_buy'] = (data['max_risk'] / (2 * data['BuyPrice_Stock2'])).round()
data['position_size2_sell'] = (data['max_risk'] / (2 * data['SellPrice_Stock2'])).round()

# Loop through the data and execute trades
for i, row in data.iterrows():
    # Z-score and trade execution logic
    z_score = row['z_score']
    sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
    sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
    current_datetime = row.name  # Use index as the datetime

    # Skip NaN prices
    if pd.isna(sell_price1) or pd.isna(buy_price2):
        continue

    # Entry and exit logic
    if z_score > z_score_threshold and current_position == 0:  # Short stock1, long stock2
        entry_price1, entry_price2 = sell_price1, buy_price2
        entry_datetime = current_datetime
        current_position = -1

    elif z_score < -z_score_threshold and current_position == 0:  # Long stock1, short stock2
        entry_price1, entry_price2 = buy_price1, sell_price2
        entry_datetime = current_datetime
        current_position = 1

    elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute == 30)):
        # Calculate PnL
        position_size1 = row['position_size1_sell'] if current_position == -1 else row['position_size1_buy']
        position_size2 = row['position_size2_buy'] if current_position == -1 else row['position_size2_sell']
        exit_price1 = buy_price1 if current_position == -1 else sell_price1
        exit_price2 = sell_price2 if current_position == -1 else buy_price2

        pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
        transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                            position_size2 * (entry_price2 + exit_price2))
        pnl -= transaction_cost
        cumulative_pnl += pnl

        tradebook.append({
            'Entry Price Stock1': entry_price1,
            'Entry Price Stock2': entry_price2,
            'Exit Price Stock1': exit_price1,
            'Exit Price Stock2': exit_price2,
            'Trade Type': 'Short' if current_position == -1 else 'Long',
            'Position Size Stock1': position_size1,
            'Position Size Stock2': position_size2,
            'PnL': round(pnl, 2),
            'Transaction Cost': round(transaction_cost, 2),
            'Portfolio Value': portfolio_value,
            'Cumulative PnL': round(cumulative_pnl, 2),
            'Entry Date': entry_datetime.date(),
            'Entry Time': entry_datetime.time(),
            'Exit Date': current_datetime.date(),
            'Exit Time': current_datetime.time(),
            'Time Difference (minutes)': (current_datetime - entry_datetime).total_seconds() / 60
        })

        current_position = 0  # Reset position

# Convert tradebook to DataFrame and save
tradebook_df = pd.DataFrame(tradebook)
tradebook_df.to_csv('tradebook.csv', index=False)

print(tradebook_df)


In [ ]:
import pandas as pd
import numpy as np
from itertools import product

# Load your dataset
data['datetime'] = pd.to_datetime(data['DateTime'])

# Parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0001  # 0.01% brokerage fee
risk_percentage = 1  # 100% risk per trade

# Different values to test
z_score_thresholds = [0.2, 0.3, 0.25]
ma_windows = [43, 430, 720, ]  # Adjust as needed
correlation_windows = ma_windows  # Adjust as needed

# Function to compute position sizes
def compute_position_sizes(data, initial_capital, risk_percentage):
    """
    Calculate position sizes based on initial capital and risk percentage.
    """
    # Calculate available capital per trade
    trade_capital = initial_capital * (risk_percentage / 100)
    
    # Calculate position sizes for buying and selling stocks
    data['position_size1_buy'] = trade_capital / data['BuyPrice_Stock1']
    data['position_size1_sell'] = trade_capital / data['SellPrice_Stock1']
    data['position_size2_buy'] = trade_capital / data['BuyPrice_Stock2']
    data['position_size2_sell'] = trade_capital / data['SellPrice_Stock2']

    return data

# Precompute rolling statistics for spread and correlation
def precompute_rolling_statistics(data, ma_windows, correlation_windows):
    # Calculate spread
    data['spread'] = (data['BuyPrice_Stock1'] + data['SellPrice_Stock1']) / (data['BuyPrice_Stock2'] + data['SellPrice_Stock2'])
    
    # Precompute rolling mean and std for spread
    for ma_window in ma_windows:
        data[f'rolling_mean_{ma_window}'] = data['spread'].rolling(window=ma_window).mean()
        data[f'rolling_std_{ma_window}'] = data['spread'].rolling(window=ma_window).std()

    # Precompute daily rolling correlation for each correlation window
    data['date'] = data['datetime'].dt.date  # Extract date for grouping
    grouped = data.groupby('date')
    for correlation_window in correlation_windows:
        data[f'correlation_{correlation_window}'] = grouped['BuyPrice_Stock1'].transform(
            lambda x: x.rolling(correlation_window).corr(data.loc[x.index, 'BuyPrice_Stock2'])
        )

    return data

# Function to calculate the strategy with different parameters
def run_backtest(data, ma_window, z_score_threshold, correlation_window):
    # Precompute rolling statistics outside the loop
    data = precompute_rolling_statistics(data, [ma_window], [correlation_window])
    data = compute_position_sizes(data, initial_capital, risk_percentage)

    # Initialize variables
    portfolio_value = initial_capital
    current_position = 0
    cumulative_pnl = 0
    portfolio_values = [portfolio_value]  # Store portfolio values for max drawdown calculation
    tradebook = []

    # Loop for trades
    for i, row in data.iterrows():
        z_score = (row['spread'] - row[f'rolling_mean_{ma_window}']) / row[f'rolling_std_{ma_window}']
        sell_price1, buy_price1 = row['SellPrice_Stock1'], row['BuyPrice_Stock1']
        sell_price2, buy_price2 = row['SellPrice_Stock2'], row['BuyPrice_Stock2']
        current_datetime = row['datetime']
        correlation = row[f'correlation_{correlation_window}']  # Correlation for the current row

        # Skip NaN prices and ensure correlation is above 50%
        if pd.isna(sell_price1) or pd.isna(buy_price2) or pd.isna(correlation) or correlation < 0.5:
            continue

        # Entry and exit logic
        if z_score > z_score_threshold and current_position == 0:  # Short stock1, long stock2
            entry_price1, entry_price2 = sell_price1, buy_price2
            entry_datetime = current_datetime
            current_position = -1

        elif z_score < -z_score_threshold and current_position == 0:  # Long stock1, short stock2
            entry_price1, entry_price2 = buy_price1, sell_price2
            entry_datetime = current_datetime
            current_position = 1

        elif current_position != 0 and (abs(z_score) < 0.01 or (current_datetime.hour == 15 and current_datetime.minute == 30)):
            # Calculate PnL
            position_size1 = row['position_size1_sell'] if current_position == -1 else row['position_size1_buy']
            position_size2 = row['position_size2_buy'] if current_position == -1 else row['position_size2_sell']
            exit_price1 = buy_price1 if current_position == -1 else sell_price1
            exit_price2 = sell_price2 if current_position == -1 else buy_price2

            pnl = (position_size1 * (exit_price1 - entry_price1)) + (position_size2 * (entry_price2 - exit_price2))
            transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) +
                                                position_size2 * (entry_price2 + exit_price2))
            pnl -= transaction_cost
            cumulative_pnl += pnl
            portfolio_value = initial_capital + cumulative_pnl
            portfolio_values.append(portfolio_value)  # Track portfolio value for max drawdown calculation

            tradebook.append({
                'Entry Price Stock1': entry_price1,
                'Entry Price Stock2': entry_price2,
                'Exit Price Stock1': exit_price1,
                'Exit Price Stock2': exit_price2,
                'Trade Type': 'Short' if current_position == -1 else 'Long',
                'Position Size Stock1': position_size1,
                'Position Size Stock2': position_size2,
                'PnL': round(pnl, 2),
                'Transaction Cost': round(transaction_cost, 2),
                'Portfolio Value': portfolio_value,
                'Cumulative PnL': round(cumulative_pnl, 2),
                'Entry Date': entry_datetime.date(),
                'Entry Time': entry_datetime.time(),
                'Exit Date': current_datetime.date(),
                'Exit Time': current_datetime.time(),
                'Time Difference (minutes)': (current_datetime - entry_datetime).total_seconds() / 60
            })

            current_position = 0  # Reset position

    # Calculate max drawdown
    max_drawdown = max((np.max(np.maximum.accumulate(portfolio_values) - portfolio_values) / np.max(np.maximum.accumulate(portfolio_values))) * 100, 0)

    # Return the results (portfolio value, cumulative PnL, and max drawdown)
    return cumulative_pnl, max_drawdown, tradebook

# Create a list of all combinations of parameters
param_combinations = list(product(z_score_thresholds, ma_windows, correlation_windows))

# Initialize a list to store the results
results = []

# Run the backtest for all parameter combinations
for z_score_threshold, ma_window, correlation_window in param_combinations:
    print(f"Running backtest with z_score_threshold={z_score_threshold}, ma_window={ma_window}, correlation_window={correlation_window}")
    cumulative_pnl, max_drawdown, tradebook = run_backtest(data.copy(), ma_window, z_score_threshold, correlation_window)
    results.append({
        'Z-Score Threshold': z_score_threshold,
        'Moving Average Window': ma_window,
        'Correlation Window': correlation_window,
        'Cumulative PnL': cumulative_pnl,
        'Max Drawdown (%)': max_drawdown
    })

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Save the results to a CSV file
results_df.to_csv('robustness_results_with_max_drawdown.csv', index=False)

print(results_df)
                pnl -= transaction_cost
                cumulative_pnl += pnl
                portfolio_value += pnl  # Update portfolio value

                # Track max drawdown
                peak_value = max(peak_value, portfolio_value)
                drawdown = (peak_value - portfolio_value) / peak_value
                max_drawdown = max(max_drawdown, drawdown)

                tradebook.append({
                    'Entry Price Stock1': entry_price1,
                    'Entry Price Stock2': entry_price2,
                    'Exit Price Stock1': exit_price1,
                    'Exit Price Stock2': exit_price2,
                    'Trade Type': 'Short' if current_position == -1 else 'Long',
                    'Position Size Stock1': posi

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Load the tradebook
tradebook = pd.read_csv('tradebook.csv')

# Parameters
num_simulations = 10000  # Number of simulations
num_trades = len(tradebook)

# Function to calculate maximum drawdown
def calculate_max_drawdown(pnl_series):
    cumulative_max = np.maximum.accumulate(pnl_series)
    drawdown = cumulative_max - pnl_series
    max_drawdown = drawdown.max()
    return max_drawdown

# Function to perform a single Monte Carlo simulation
def simulate_cumulative_pnl():
    sampled_trades = tradebook.sample(n=num_trades, replace=True, random_state=None)
    pnl = sampled_trades['PnL'].cumsum().values
    return pnl

# Measure start time
start_time = time.time()

# Run simulations in parallel
cumulative_pnl_simulations = Parallel(n_jobs=-1)(
    delayed(simulate_cumulative_pnl)() for _ in range(num_simulations)
)

# Measure end time
end_time = time.time()

# Convert simulation results to a DataFrame
cumulative_pnl_df = pd.DataFrame(cumulative_pnl_simulations).T
cumulative_pnl_df.columns = [f"Sim_{i+1}" for i in range(num_simulations)]

# Calculate mean cumulative PnL as the trading system line
mean_cumulative_pnl = cumulative_pnl_df.mean(axis=1)

# Compute specifics for Monte Carlo simulation
final_profits = cumulative_pnl_df.iloc[-1]  # Final PnL for each simulation
mean_profit = final_profits.mean()
std_dev_profit = final_profits.std()
max_drawdowns = cumulative_pnl_df.apply(calculate_max_drawdown, axis=0)
mean_drawdown = max_drawdowns.mean()
success_rate = (final_profits > 0).mean() * 100  # Percentage of positive outcomes

# Define success criteria
is_successful = mean_profit > 0 and mean_drawdown < abs(mean_profit) * 2

# Print specifics
print("Monte Carlo Simulation Specifics:")
print(f"Number of Simulations: {num_simulations}")
print(f"Number of Trades Per Simulation: {num_trades}")
print(f"Mean Final Profit: {mean_profit:.2f}")
print(f"Standard Deviation of Profit: {std_dev_profit:.2f}")
print(f"Mean Maximum Drawdown: {mean_drawdown:.2f}")
print(f"Success Rate (% of Positive Simulations): {success_rate:.2f}%")
print(f"Monte Carlo Simulation Passed: {'Successful' if is_successful else 'Failed'}")

# Visualization
plt.figure(figsize=(14, 7))

# Use a colormap for light colors
colors = cm.get_cmap("Pastel1", num_simulations)

# Plot each simulation with a different light color
for i, col in enumerate(cumulative_pnl_df.columns):
    plt.plot(cumulative_pnl_df[col], color=colors(i / num_simulations), linewidth=0.7, alpha=0.8)

# Highlight the mean trading system line
plt.plot(mean_cumulative_pnl, color="blue", linewidth=2, label="Mean Trading System Line")

# Add labels and title
plt.title("Monte Carlo Simulation: Profit Over Time", fontsize=14)
plt.xlabel("Trades", fontsize=12)
plt.ylabel("Cumulative Profit", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)

# Show the plot
plt.show()

# Print execution time
print(f"Simulation completed in {end_time - start_time:.2f} seconds.")


In [ ]:
import matplotlib.pyplot as plt

# Ensure 'Entry Date' column is in datetime format
tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])

# Extract the weekday from the 'Entry Date' column
tradebook_df['Weekday'] = tradebook_df['Entry Date'].dt.day_name()

# Group by weekday and sum the PnL
weekday_pnl = tradebook_df.groupby('Weekday')['PnL'].sum()

# Sort by weekday order
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_pnl = weekday_pnl.reindex(weekday_order)

# Plot the histogram
plt.figure(figsize=(10, 6))
weekday_pnl.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('PnL Distribution by Weekday', fontsize=16)
plt.xlabel('Weekday', fontsize=14)
plt.ylabel('Total PnL', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])

# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()

# Calculate drawdown
tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']

# Create the plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the cumulative PnL
ax.plot(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='blue')

# Fill the drawdown area with red color
ax.fill_between(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], tradebook_df['Max Cumulative PnL'], where=(tradebook_df['Cumulative PnL'] < tradebook_df['Max Cumulative PnL']),
                facecolor='red', alpha=0.3, label='Drawdown')

# Add labels and title
ax.set_xlabel('Exit Date')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Cumulative PnL and Drawdown Over Time')
ax.legend()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])

# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()

# Calculate the maximum cumulative PnL up to each point
tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()

# Calculate the drawdown as the difference between cumulative PnL and the max cumulative PnL
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']

# Calculate the average drawdown
average_drawdown = tradebook_df['Drawdown'].mean()

# Create the plot
plt.figure(figsize=(12, 6))

# Plot the drawdown
plt.plot(tradebook_df['Exit Date'], tradebook_df['Drawdown'], label='Drawdown', color='red', linestyle='-', linewidth=2)

# Plot the average drawdown as a horizontal blue line
plt.axhline(y=average_drawdown, color='blue', linestyle='--', label=f'Average Drawdown ({average_drawdown:.2f})')

# Add labels and title
plt.xlabel('Exit Date')
plt.ylabel('Drawdown')
plt.title('Drawdown Over Time')
plt.legend()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Create a histogram of PnL distribution
plt.figure(figsize=(14, 7))  # Set the size of the plot
plt.hist(tradebook_df['PnL'], bins=50, color='skyblue', edgecolor='black')

# Add labels and title
plt.xlabel('PnL per Trade')
plt.ylabel('Frequency')
plt.title('PnL Distribution of Trades')

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors

# Load your tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Entry Date' to datetime format
tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])

# Extract Year and Month from 'Entry Date'
tradebook_df['Year'] = tradebook_df['Entry Date'].dt.year
tradebook_df['Month'] = tradebook_df['Entry Date'].dt.month

# Get the first trade of each month for the portfolio value
first_trade_of_month = tradebook_df.groupby(['Year', 'Month']).first().reset_index()

# Merge the first trade portfolio value with the main data
tradebook_df = tradebook_df.merge(first_trade_of_month[['Year', 'Month', 'Portfolio Value']], 
                                   on=['Year', 'Month'], suffixes=('', '_FirstTrade'))

# Calculate the Monthly Profit Percentage
tradebook_df['Monthly Profit %'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value_FirstTrade']) * 100

# Group by Year and Month to get the sum of monthly profit percentages
monthly_profit_df = tradebook_df.groupby(['Year', 'Month'])['Monthly Profit %'].sum().reset_index()

# Pivot the data for the heatmap
heatmap_data = monthly_profit_df.pivot_table(values='Monthly Profit %', index='Year', columns='Month', aggfunc='sum')
heatmap_data.fillna(0, inplace=True)

# Create the custom colormap for positive (green) and negative (red) values
cmap = mcolors.ListedColormap(['#ff6666', '#3BB143'])  # Red for negative, Green for positive
norm = mcolors.TwoSlopeNorm(vmin=heatmap_data.min().min(), vcenter=0, vmax=heatmap_data.max().max())

# Create the heatmap plot
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap=cmap, norm=norm, cbar_kws={'label': 'Monthly Profit %'}, linewidths=0.5)

# Set plot labels and title
plt.title("Monthly Profit Percentage Heatmap")
plt.xlabel("Month")
plt.ylabel("Year")
plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

# Tight layout for better presentation
plt.tight_layout()

# Show the plot on screen
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])

# Calculate yearly PnL by summing PnL for each year
tradebook_df['Year'] = tradebook_df['Exit Date'].dt.year
yearly_returns = tradebook_df.groupby('Year')['PnL'].sum().reset_index()

# Create a bar chart for yearly returns using Matplotlib
plt.figure(figsize=(14, 6))  # Set the figure size
plt.bar(yearly_returns['Year'], yearly_returns['PnL'], color='skyblue')

# Add title and labels
plt.title('Yearly Returns', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Profit', fontsize=14)

# Show the bar chart
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Exit Date' to datetime format if it's not already
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])

# Add a column for cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()

# Calculate quarterly periods based on the exit date
tradebook_df['Quarter'] = tradebook_df['Exit Date'].dt.to_period('Q')

# Group by quarter and calculate the total profit (sum of PnL) for each quarter
quarterly_profit = tradebook_df.groupby('Quarter')['PnL'].sum()

# Calculate the average quarterly profit
average_profit = quarterly_profit.mean()

# Plot the quarterly profit with grid enabled
plt.figure(figsize=(12, 8))  # Increase figure size

# Plotting the total profit for each quarter
quarterly_profit.plot(kind='bar', color='skyblue', edgecolor='black')

# Add a horizontal line showing the average in red
plt.axhline(y=average_profit, color='red', linestyle='--', label=f'Average: {average_profit:.2f}')

# Add labels and title
plt.xlabel('Quarter', fontsize=14)
plt.ylabel('Quarterly Profit', fontsize=14)
plt.title('Quarterly Profit', fontsize=16)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=12)

# Show grid for better visualization
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

# Add a legend for the average line
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors
from fpdf import FPDF
import numpy as np

# Load the tradebook data from the CSV file
tradebook_df = pd.read_csv('tradebook.csv')

# Create a FPDF object for PDF generation
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# Function to save the plot as an image
def save_plot_as_image(plot_function, filename, *args):
    plt.figure(figsize=(12, 6))  # Set the size of the plot
    plot_function(*args)  # Call the plotting function with arguments
    plt.tight_layout()  # Adjust layout to prevent clipping
    plt.savefig(filename, format='png')  # Save as PNG
    plt.close()  # Close the plot to avoid it being displayed in the notebook

# Plot 1: Cumulative PnL and Drawdown Over Time
def plot_cumulative_pnl_and_drawdown(tradebook_df):
    tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
    tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()
    tradebook_df['Max Cumulative PnL'] = tradebook_df['Cumulative PnL'].cummax()
    tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='blue')
    ax.fill_between(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], tradebook_df['Max Cumulative PnL'], where=(tradebook_df['Cumulative PnL'] < tradebook_df['Max Cumulative PnL']),
                    facecolor='red', alpha=0.3, label='Drawdown')
    ax.set_xlabel('Exit Date')
    ax.set_ylabel('Cumulative PnL')
    ax.set_title('Cumulative PnL and Drawdown Over Time')
    ax.legend()
    plt.xticks(rotation=45)

# Save the plot and add to PDF
save_plot_as_image(plot_cumulative_pnl_and_drawdown, "cumulative_pnl_and_drawdown.png", tradebook_df)
pdf.add_page()
pdf.image("cumulative_pnl_and_drawdown.png", x=10, y=10, w=190)

# Plot 2: Drawdown Over Time
def plot_drawdown(tradebook_df):
    tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Max Cumulative PnL']
    average_drawdown = tradebook_df['Drawdown'].mean()

    plt.figure(figsize=(12, 6))
    plt.plot(tradebook_df['Exit Date'], tradebook_df['Drawdown'], label='Drawdown', color='red', linestyle='-', linewidth=2)
    plt.axhline(y=average_drawdown, color='blue', linestyle='--', label=f'Average Drawdown ({average_drawdown:.2f})')
    plt.xlabel('Exit Date')
    plt.ylabel('Drawdown')
    plt.title('Drawdown Over Time')
    plt.legend()
    plt.xticks(rotation=45)

# Save the plot and add to PDF
save_plot_as_image(plot_drawdown, "drawdown.png", tradebook_df)
pdf.add_page()
pdf.image("drawdown.png", x=10, y=10, w=190)

# Plot 3: PnL Distribution of Trades
def plot_pnl_distribution(tradebook_df):
    plt.figure(figsize=(14, 7))
    plt.hist(tradebook_df['PnL'], bins=50, color='skyblue', edgecolor='black')
    plt.xlabel('PnL per Trade')
    plt.ylabel('Frequency')
    plt.title('PnL Distribution of Trades')

# Save the plot and add to PDF
save_plot_as_image(plot_pnl_distribution, "pnl_distribution.png", tradebook_df)
pdf.add_page()
pdf.image("pnl_distribution.png", x=10, y=10, w=190)

# Plot 4: Monthly Profit Heatmap
def plot_monthly_profit_heatmap(tradebook_df):
    # Convert 'Entry Date' to datetime format
    tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])
    
    # Extract Year and Month from 'Entry Date'
    tradebook_df['Year'] = tradebook_df['Entry Date'].dt.year
    tradebook_df['Month'] = tradebook_df['Entry Date'].dt.month
    
    # Get the first trade of each month for the portfolio value
    first_trade_of_month = tradebook_df.groupby(['Year', 'Month']).first().reset_index()
    
    # Merge the first trade portfolio value with the main data
    tradebook_df = tradebook_df.merge(first_trade_of_month[['Year', 'Month', 'Portfolio Value']], 
                                       on=['Year', 'Month'], suffixes=('', '_FirstTrade'))
    
    # Calculate the Monthly Profit Percentage
    tradebook_df['Monthly Profit %'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value_FirstTrade']) * 100
    
    # Group by Year and Month to get the sum of monthly profit percentages
    monthly_profit_df = tradebook_df.groupby(['Year', 'Month'])['Monthly Profit %'].sum().reset_index()
    
    # Pivot the data for the heatmap
    heatmap_data = monthly_profit_df.pivot_table(values='Monthly Profit %', index='Year', columns='Month', aggfunc='sum')
    heatmap_data.fillna(0, inplace=True)
    
    # Create the custom colormap for positive (green) and negative (red) values
    cmap = mcolors.ListedColormap(['#ff6666', '#3BB143'])  # Red for negative, Green for positive
    norm = mcolors.TwoSlopeNorm(vmin=heatmap_data.min().min(), vcenter=0, vmax=heatmap_data.max().max())
    
    # Create the heatmap plot
    plt.figure(figsize=(14, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap=cmap, norm=norm, cbar_kws={'label': 'Monthly Profit %'}, linewidths=0.5)
    
    # Set plot labels and title
    plt.title("Monthly Profit Percentage Heatmap")
    plt.xlabel("Month")
    plt.ylabel("Year")
    plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
    
    # Tight layout for better presentation
    plt.tight_layout()

# Save the plot and add to PDF
save_plot_as_image(plot_monthly_profit_heatmap, "monthly_profit_heatmap.png", tradebook_df)
pdf.add_page()
pdf.image("monthly_profit_heatmap.png", x=10, y=10, w=190)

def save_plot_as_image(plot_function, filename, *args):
    plt.figure(figsize=(12, 6))  # Set the size of the plot
    plot_function(*args)  # Call the plotting function with arguments
    plt.tight_layout()  # Adjust layout to prevent clipping
    plt.savefig(filename, format='png')  # Save as PNG
    plt.close()  # Close the plot to avoid it being displayed in the notebook

    # Plot 1: PnL Distribution by Weekday
    tradebook_df['Weekday'] = tradebook_df['Entry Date'].dt.day_name()
    
    # Group by weekday and sum the PnL
    weekday_pnl = tradebook_df.groupby('Weekday')['PnL'].sum()
    
    # Sort by weekday order
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_pnl = weekday_pnl.reindex(weekday_order)
    
# Save and add to the PDF
save_plot_as_image(lambda: weekday_pnl.plot(kind='bar', color='skyblue', edgecolor='black'),
                       "weekday_pnl_distribution.png")
pdf.add_page()
pdf.image("weekday_pnl_distribution.png", x=10, y=10, w=190)

# Save the PDF to a file
pdf.output("tradebook_charts.pdf")

print("PDF has been created successfully!")
